## Task: Predict Carbon Emissions Based on Satellite Observations

* You are provided with a time series satellite observations dataset extracted from Sentinel-5P satelite from 2019 to 2021.

* Your objective is to build a regression model that accurately predicts the amount of carbon emissions in the next years.

* Your target is the column: "emission".

* You are provided with the code to download and load the csv file.

* Your work will be evaluated based on the completion of the given tasks below.

* You are allowed to use any models or libraries you want.
    
---


In [ ]:
import pandas as pd
import gdown
import kagglehub
import os

In [ ]:

path = kagglehub.dataset_download("mohammad2012191/q3-data")

print("Path to dataset files:", path)

In [ ]:
csv_path = os.path.join(path, "train.csv")

data = pd.read_csv(csv_path)
data.head()

# EDA & Preprocessing

1. Drop the ID feature (ID_LAT_LON_YEAR_WEEK):

In [ ]:
# 1. TODO
data = data.drop('ID_LAT_LON_YEAR_WEEK',axis=1)

In [ ]:
data

2. Check for the missing values and handle them.


In [ ]:
# 2. TODO
# data.isna().sum()
data.info()

In [ ]:
data = data.fillna(data.mean())

3. Plot the "emission" histogram.

In [ ]:
# 3. TODO
data.emission.hist(bins=30)

4. Plot the "latitude" and "longitude" using a scatter plot then colorize the points using the "emission" column.

In [ ]:
# 4. TODO
import matplotlib.pyplot as plt
plt.scatter(data["latitude"],data["longitude"],c=data["emission"])

# Feature engineering

1. Add a new feature representing "Location" (To do that, you should convert "longitude" and "latitude" features to string type, concatenate them and add the result as a new feature).

In [ ]:
# 1. TODO
data["location"] = data["longitude"].astype(str) + data["latitude"].astype(str)

In [ ]:
data

2. Add one aggregation feature representing the average emissions per location (You should groupby "Location" feature and take mean of the "emission", then merge the result to the data)

In [ ]:
# 2. TODO
gp = data.groupby("location").mean("emission")
pd.merge(data,gp,on='location', how="left")

3. Use Label encoder to encode all the categorical features

In [ ]:
# 3. TODO
from sklearn.preprocessing import LabelEncoder
categ_feats = data.select_dtypes("object").columns
for cat in categ_feats:
  le = LabelEncoder()
  data[cat] = le.fit_transform(data[cat])

In [ ]:
data

# Modeling

### Baseline:

In [ ]:
X = data.drop('emission',axis=1)
y = data['emission']

1. Create a baseline using the "emission" median and calculate MAE score.

In [ ]:
# 1. TODO
import numpy as np
from sklearn.metrics import mean_absolute_error
baseline = np.full_like(y,y.median())
mean_absolute_error(baseline,y)

2. Perform a Time-Based Train-Test Split:
  * You will use the "year" column to split data manually using pandas.
  * Use all samples from 2019 and 2020 as the training set.
  * Use all samples from 2021 as the validation set.
  * Construct X_train, X_valid, y_train, y_valid

In [ ]:
X.year.value_counts()

In [ ]:
# 2. TODO
X_train = X[X["year"] < 2021]
X_valid = X[X["year"] == 2021]
y_train = y[X_train.index]
y_valid = y[X_valid.index]

3. Train a LGBMRegressor on the training data.

In [ ]:
# 3. TODO
from lightgbm import LGBMRegressor
model = LGBMRegressor(max_depth=6)
model.fit(X_train,y_train)


4. Evaluate the Model on the validation data Using MAE

In [ ]:
# 4. TODO
y_pred = model.predict(X_valid)
mean_absolute_error(y_pred,y_valid)

5. Plot the features importance of your model.

In [ ]:
# 5. TODO
# Feature importance
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

6. Plot the validation predictions using a histogram.

In [ ]:
# 6. TODO
pd.Series(y_pred).hist()